# EEG motor imagery: from raw signals to a hyperparameter-tuned network 🧠
Can a computer tell whether you are imagining moving your **left** or your **right** hand, only from your EEG, if it has **never seen you before**? Today you build the whole pipeline, step by step.

**How this works** — you will not write code. In each step you:

| | |
|---|---|
| 🤔 **Predict** | choose what you *expect* before running |
| 🎛️ **Decide** | change the controls on the right of a cell (sliders, menus) |
| ▶️ **Run** | press the ▶ button of the cell (or `Shift + Enter`) |
| 👀 **Look** | read the figure with the explanation below it |
| 💬 **Discuss** | talk it through with your group or neighbour |

Code cells are folded. If you are curious, double-click a cell title to see its code.

**How this notebook is organised**
- Everything is in **one notebook**, split in **8 parts** (00–07). Each part is **folded**: click the ▸ arrow next to its title to open it, only when we get there. The ☰ table of contents on the left lets you jump between parts.
- Run the parts **in order**: each one uses what the previous ones computed.
- 🆘 **Arrived late, or the runtime disconnected?** Click the first cell of the part we are doing and press **Ctrl + F8** (*Runtime → Run before*). Everything above runs with the default choices and you catch up in a couple of minutes.
- 📊 **Live class poll:** your answers in the 🤔 and 📤 cells (and your 🏆 results) are sent **anonymously** to a poll that the teacher projects. Only the answer and a random nickname (like *Swift Otter 42*) are sent — no name, no email. You can switch it off in the setup cell.

# 00 · Setup ⚙️ — run this first
This notebook asks Colab for a **GPU** automatically. Check it: **Runtime → Change runtime type** should say **T4 GPU** (if no GPU is available, CPU also works: the training cells have a `use_precomputed` box).

Press ▶ on the cell below. It installs two libraries, downloads the helper file and the dataset (~215 MB) and checks the GPU. It takes about 1 minute.

In [ ]:
#@title ⚙️ Setup — run me first (≈1 min)
share_answers = True #@param {type:"boolean"}
%pip install -q mne keras-tuner
import os, importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/ptorija/eeg-motor-imagery-practical/main/bci_utils.py", "bci_utils.py")   # always the latest version
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import bci_utils
from bci_utils import *
bci_utils.SHARE = share_answers
setup_style()
data = load_data()
X, y, subject, fs = data["X"], data["y"], data["subject"], data["fs"]
import tensorflow as tf
print("GPU ✅ — training will be fast" if tf.config.list_physical_devices("GPU")
      else "⚠️ No GPU: Runtime → Change runtime type → T4 GPU (or tick use_precomputed in the training cells)")
print("✅ Ready." + (f" Your anonymous nickname for the class poll: {nickname()}" if share_answers else " (class poll switched off)"))

In [ ]:
#@title 🔄 Update the helper file — only if the teacher asks you to
urllib.request.urlretrieve("https://raw.githubusercontent.com/ptorija/eeg-motor-imagery-practical/main/bci_utils.py", "bci_utils.py")
importlib.reload(bci_utils)
from bci_utils import *
bci_utils.SHARE = share_answers
print("✅ Helpers updated — carry on where you were")

# 01 · Meet the data 🧠
**Goal:** understand what was recorded, how the data is organised, and whether the difference between *left* and *right* is visible by eye.

## 1. The experiment
**PhysioNet EEG Motor Movement/Imagery dataset** (Schalk et al., 2004). 109 volunteers wore a 64-electrode EEG cap.
A target appeared on the **left** or **right** of a screen and they had to **imagine** opening and closing that fist — *without moving* — until the target disappeared (~4 s).

This is the classic task of a **motor-imagery brain–computer interface (BCI)**: control a computer only by imagining movements.

We prepared the data for you: each **trial** is the 4 seconds after the target appears, recorded at **160 Hz** (160 numbers per second) → **640 samples × 64 electrodes**.

In [ ]:
#@title 👀 One trial, on a timeline
fig, ax = plt.subplots(figsize=(9, 1.8))
ax.axvspan(-1, 0, color="#eee"); ax.text(-0.5, 0.5, "rest", ha="center", va="center")
ax.axvspan(0, 4, color=COLORS["left"], alpha=0.25); ax.text(2, 0.5, "imagine LEFT or RIGHT fist\n(our trial: 4 s = 640 samples)", ha="center", va="center")
ax.axvspan(4, 5.5, color="#eee"); ax.text(4.75, 0.5, "rest", ha="center", va="center")
ax.annotate("target appears", (0, 1), (0.3, 1.25), arrowprops=dict(arrowstyle="->"), annotation_clip=False)
ax.set(xlim=(-1, 5.5), ylim=(0, 1), yticks=[], xlabel="time (s)"); ax.grid(False)
plt.show()

## 2. How much data do we have?
🤔 **Predict:** is there roughly the same number of *left* and *right* trials? Does every person have the same number of trials?

In [ ]:
#@title 🤔 Your prediction
balanced = "I think yes" #@param ["I think yes", "I think no", "No idea"]
print("Saved your prediction:", balanced)
share("balanced", balanced)

In [ ]:
#@title ▶️ Count the trials
n_trials, n_ch, n_t = X.shape
print(f"Trials: {n_trials}   Electrodes: {n_ch}   Samples per trial: {n_t} ({n_t/fs:.0f} s)   People: {len(np.unique(subject))}")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
counts = [np.sum(y == 0), np.sum(y == 1)]
axs[0].bar(["left", "right"], counts, color=[COLORS["left"], COLORS["right"]])
for i, c in enumerate(counts): axs[0].text(i, c, f"{c}\n({c/n_trials:.0%})", ha="center", va="bottom")
axs[0].set(title="Trials per class", ylabel="trials", ylim=(0, max(counts) * 1.25))
per_subject = np.bincount(subject)[np.unique(subject)]
axs[1].hist(per_subject, bins=np.arange(per_subject.min() - 1.5, per_subject.max() + 2.5), color="#4C72B0")
axs[1].set(title="Trials per person", xlabel="number of trials", ylabel="people")
plt.tight_layout(); plt.show()

👀 **What am I looking at?** Left: the number of trials of each class. Right: how many trials each person contributed.

**What should you notice?** The classes are almost perfectly **balanced** (~50/50), so a model that always says "left" would be right only ~50% of the time. This is our **chance level**: any useful model must be clearly above 50%. Almost everybody has 45 trials (3 runs × 15 targets).

## 3. Where are the electrodes? (MNE)
[MNE-Python](https://mne.tools) is the standard Python library for EEG/MEG. Here we use it only to draw the electrode positions.

In [ ]:
#@title ▶️ Plot the 64 electrodes with MNE
import mne
mne.set_log_level("ERROR")
info = mne_info(data)
fig = mne.viz.plot_sensors(info, kind="topomap", show_names=["C3", "Cz", "C4", "Fz", "Pz", "Oz"],
                           title="64 EEG electrodes (seen from above, nose up)", show=False)
fig.set_size_inches(4.5, 4.5); plt.show()

👀 **What should you notice?** **C3** sits over the **left** motor cortex and **C4** over the **right** motor cortex.
The motor cortex is *crossed*: the **left** hemisphere controls the **right** hand and vice versa. Remember this — we will look for it in part 03.

## 4. What does a trial look like?
🎛️ **Decide:** pick a trial with the slider and run the cell. Try several.

In [ ]:
#@title 🎛️ Look at one trial
trial = 0 #@param {type:"slider", min:0, max:4766, step:1}
channels = ["FC3", "FCz", "FC4", "C3", "Cz", "C4", "CP3", "CPz", "CP4"]
t = np.arange(X.shape[-1]) / fs
fig, ax = plt.subplots(figsize=(9, 6))
offset = 0
for ch in channels:
    sig = X[trial, data["ch_names"].index(ch)]
    ax.plot(t, sig - sig.mean() + offset, color=COLORS[CLASS_NAMES[y[trial]]], lw=0.8)
    ax.text(-0.08, offset, ch, ha="right", va="center", fontweight="bold")
    offset -= 100
ax.set(xlim=(-0.45, 4.4), yticks=[], xlabel="time (s)", title=f"Trial {trial} — person {subject[trial]} imagined the {CLASS_NAMES[y[trial]].upper()} hand")
ax.plot([4.05, 4.05], [0, -50], "k", lw=2); ax.text(4.1, -25, "50 µV", va="center")
plt.show()

👀 **What am I looking at?** Nine electrodes around the motor cortex, one trial, 4 seconds. The scale bar shows 50 microvolts (µV). EEG is **tiny** (a phone battery is ~100 000× larger) and full of rhythms and noise.

🤔 **Predict:** if we **average** hundreds of *left* trials and hundreds of *right* trials, will the two averages look different at C3 and C4?

In [ ]:
#@title 🤔 Your prediction
averages_differ = "Yes, clearly" #@param ["Yes, clearly", "A little", "No"]
share("averages_differ", averages_differ)

In [ ]:
#@title ▶️ Average signal: left vs right
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, ch in zip(axs, ["C3", "C4"]):
    k = data["ch_names"].index(ch)
    for c in (0, 1):
        sig = X[y == c, k]
        sig = sig - sig.mean(-1, keepdims=True)
        ax.plot(t, sig.mean(0), color=COLORS[CLASS_NAMES[c]], label=CLASS_NAMES[c])
    ax.set(title=f"Average of all trials — {ch}", xlabel="time (s)")
axs[0].set_ylabel("µV"); axs[0].legend()
plt.tight_layout(); plt.show()

👀 **What should you notice?**
- After ~1 s the two averages are almost flat and very similar: brain rhythms are not synchronised with the target, so averaging cancels them. The information about imagined movements is in the **power of the rhythms**, not in the shape of the signal (parts 02–03).
- **But** at **C3** there is a clear difference in the **first second** (a peak around 0.3 s). Strange: the motor cortex should not produce a big, fast, synchronised wave when you *imagine* a movement...

🕵️ **Keep this mystery in mind** — we solve it in part 03.

💬 **Discuss:** what else happens in the first second after a target appears on the left or on the right of a screen?

## ✅ What you learned
- 4767 trials, 106 people, 64 electrodes, 4 s at 160 Hz. Classes balanced → **chance level ≈ 50%**.
- C3 = left motor cortex (right hand), C4 = right motor cortex (left hand).
- The motor-imagery difference is **not visible** in the raw averages → we need preprocessing and features.
- 🕵️ Open mystery: an early left/right difference at C3 in the first second.

# 02 · Preprocessing 🧹
**Goal:** decide how to clean the signal before giving it to a model: which **frequencies** to keep, what to do with **artifacts**, and how to put all trials on the **same scale**.

In [ ]:
#@title (helper) MNE objects for this notebook
import mne
mne.set_log_level("ERROR")
info = mne_info(data)
print("MNE info ready:", len(info.ch_names), "channels at", info["sfreq"], "Hz")

## 1. Which frequencies are in the signal? (MNE)
A **power spectrum** shows how much energy the signal has at each frequency. We compute it with MNE on 500 trials.

In [ ]:
#@title ▶️ Power spectrum with MNE
epochs = mne.EpochsArray(X[:500] * 1e-6, info, verbose=False)   # MNE works in volts
spectrum = epochs.compute_psd(method="welch", fmin=1, fmax=79, n_fft=fs)
fig = spectrum.plot(average=False, amplitude=False, show=False)
fig.set_size_inches(9, 3.6); fig.axes[0].set_title("Power spectrum of each electrode (500 trials)")
plt.show()

👀 **What am I looking at?** One line per electrode. x-axis: frequency (Hz). y-axis: power (in decibels, a logarithmic scale).

**What should you notice?**
- Power is very high at **low frequencies** (< 4 Hz): slow drifts, eye movements, sweat → mostly **noise** for us.
- There is a bump around **8–13 Hz**: the **mu/alpha rhythm**. Over the motor cortex it is called **mu**, and it is the rhythm that changes when you imagine a movement.
- **Beta** (13–30 Hz) also relates to movement.
- A very sharp, narrow peak at **60 Hz**. That is not the brain: it is the **power line**. This dataset was recorded in the USA, where the electricity grid oscillates at 60 Hz (in Europe it would be at **50 Hz**). It is tiny in time but huge in the spectrum.
- Above ~30 Hz the brain signal is weak, and what remains is mostly **muscle** activity (jaw, neck) and noise.
- The data stops at **80 Hz** because it is sampled at 160 Hz: the highest frequency you can represent is **half the sampling rate** (the *Nyquist frequency*).

## 2. 🎛️ Decide: band-pass filter
A **band-pass filter** keeps only the frequencies between a low and a high cut-off.

🤔 Before moving the sliders: which range would you keep to capture mu and beta but remove slow drifts?

In [ ]:
#@title 🎛️ Choose the filter and compare
low_cut = 8 #@param {type:"slider", min:1, max:12, step:1}
high_cut = 30 #@param {type:"slider", min:15, max:75, step:1}
trial = 10 #@param {type:"integer"}
c3 = data["ch_names"].index("C3")
Xf_demo = bandpass(X[:500], low_cut, high_cut, fs)
f, P_raw = psd(X[:500, c3]); _, P_f = psd(Xf_demo[:, c3])
t = np.arange(X.shape[-1]) / fs
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
axs[0].semilogy(f, P_raw.mean(0), color=GREY, label="raw")
axs[0].semilogy(f, P_f.mean(0), color="#4C72B0", label=f"filtered {low_cut}–{high_cut} Hz")
axs[0].axvspan(8, 13, color="gold", alpha=0.3, label="mu (8–13 Hz)")
axs[0].axvspan(13, 30, color="orange", alpha=0.12, label="beta (13–30 Hz)")
axs[0].axvline(60, color="#C44E52", ls=":", lw=1.5, label="power line (60 Hz)")
axs[0].set(xlabel="frequency (Hz)", ylabel="power (µV²/Hz, log)", title="Spectrum at C3"); axs[0].legend(fontsize=9)
axs[1].plot(t, X[trial, c3] - X[trial, c3].mean(), color=GREY, lw=0.8, label="raw")
axs[1].plot(t, Xf_demo[trial, c3], color="#4C72B0", lw=1, label="filtered")
axs[1].set(xlabel="time (s)", ylabel="µV", title=f"Trial {trial}, electrode C3"); axs[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

📊 Played with the sliders? Now tell the class which band **you** would keep for motor imagery 👇

In [ ]:
#@title 📤 Your choice: which band would you keep? (shared anonymously)
my_low = 8 #@param {type:"slider", min:1, max:12, step:1}
my_high = 30 #@param {type:"slider", min:15, max:75, step:5}
share("filter_low", f"{my_low} Hz")
share("filter_high", f"{my_high} Hz")

👀 **What should you notice?** Left: the filter removes the frequencies outside your range (the blue line drops outside it). Right: the filtered trial no longer wanders up and down slowly; only the oscillations remain.

💬 **Discuss:** what happens if you set `high_cut = 15`? You lose beta. And `low_cut = 12`? You cut the mu rhythm in half. Try `1–75 Hz` and look at the trial: the slow waves come back, and in the spectrum the **60 Hz power-line peak** is back too. Try `8–55 Hz` vs `8–65 Hz`: one keeps the power line out, the other lets it in.

For the rest of the session we use **8–30 Hz**, the classic motor-imagery band (did the class agree? 📊): it keeps exactly mu + beta, and removes slow drifts and most eye artifacts (you will see why this matters in part 03).

## 3. 🎛️ Decide: what do we do with artifacts?
Artifacts are big signals that do not come from the brain: blinks and **eye movements**, jaw clenching, a loose electrode... A common rule is to **reject** trials whose amplitude is too large.

🤔 **Predict:** with the rule "reject a trial if *any* of its 64 electrodes exceeds 150 µV peak-to-peak", what fraction of trials will we **keep**?

In [ ]:
#@title 🤔 Your prediction
kept_prediction = "about 90%" #@param ["about 90%", "about 70%", "about 45%", "about 20%"]
share("kept_prediction", kept_prediction)

In [ ]:
#@title 🎛️ Choose a rejection threshold
threshold_uV = 150 #@param {type:"slider", min:50, max:1000, step:25}
Xf = bandpass(X, 8, 30, fs)
ptp = peak_to_peak(Xf)
keep = ptp < threshold_uV
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6), gridspec_kw=dict(width_ratios=[1.2, 1, 1]))
axs[0].hist(ptp, bins=np.logspace(1.5, 3.2, 60), color="#4C72B0")
axs[0].axvline(threshold_uV, color="red", lw=2); axs[0].set_xscale("log")
axs[0].set(title=f"Kept: {keep.mean():.0%} of trials", xlabel="largest peak-to-peak in the trial (µV, log)", ylabel="trials")
per_subj = np.array([keep[subject == s].mean() for s in np.unique(subject)])
axs[1].hist(per_subj * 100, bins=20, color="#55A868")
axs[1].set(title="Trials kept per person", xlabel="% of the person's trials kept", ylabel="people")
bad = np.where(~keep)[0]
k = bad[np.argmax(ptp[bad])] if len(bad) else 0
worst_ch = int(np.argmax(Xf[k].max(-1) - Xf[k].min(-1)))
t = np.arange(X.shape[-1]) / fs
axs[2].plot(t, Xf[k, worst_ch], color="red", lw=0.8, label=f"rejected (trial {k}, {data['ch_names'][worst_ch]})")
good = np.where(keep)[0]
if len(good): axs[2].plot(t, Xf[good[0], worst_ch], color="#4C72B0", lw=0.8, label="kept trial, same electrode")
axs[2].set(title="Example", xlabel="time (s)", ylabel="µV"); axs[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"With {threshold_uV} µV we keep {keep.sum()} of {len(keep)} trials ({keep.mean():.0%}). "
      f"{np.sum(per_subj < 0.5)} people would lose more than half of their data.")

👀 **What am I looking at?** Left: the size of the largest wave in each trial; everything right of the red line is rejected. Middle: how much data each person keeps. Right: the worst rejected trial next to a normal one.

**What should you notice?** With 64 electrodes, almost every trial has *some* electrode with a large wave. A strict threshold throws away most of the data, and some people lose almost everything.

💬 **Discuss:** is it better to have *few clean* trials or *many noisy* ones? For deep learning we usually prefer more data plus a robust scaling step (next section). In this session we **keep all trials**.

## 4. Same scale for every trial: standardisation
People differ: some have thicker skulls or better electrode contact, so their signals are larger. We rescale **each electrode of each trial** to mean 0 and standard deviation 1 (and clip extreme values).

Important: this uses **only the trial itself**, so no information travels between the training and the test data.

In [ ]:
#@title ▶️ Before vs after standardisation
Xs = standardize_trials(Xf[:500])
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
for s_ in np.unique(subject[:500])[:4]:
    m = subject[:500] == s_
    axs[0].hist(Xf[:500][m].ravel(), bins=np.linspace(-80, 80, 80), histtype="step", lw=1.5, label=f"person {s_}", density=True)
    axs[1].hist(Xs[m].ravel(), bins=np.linspace(-5, 5, 80), histtype="step", lw=1.5, density=True)
axs[0].set(title="Filtered (µV)", xlabel="value"); axs[0].legend(fontsize=8)
axs[1].set(title="Standardised (no units)", xlabel="value")
plt.tight_layout(); plt.show()

👀 **What should you notice?** Before: each person has a different spread of values. After: everybody is on the same scale, which makes training much easier for neural networks.

## ✅ What you learned — our preprocessing pipeline
**raw EEG → band-pass 8–30 Hz → keep all trials → standardise each trial → model**  (one more step will be added in part 03 🕵️)

In [ ]:
#@title 🧹 Free memory before the next part
import gc
for _v in ['Xf', 'Xf_demo', 'Xs', 'ptp']:
    globals().pop(_v, None)
gc.collect(); print("✅ Memory freed")

# 03 · Brain… or eyes? 🕵️
**Goal:** (1) find the brain signature of imagining a hand movement — the **event-related desynchronisation (ERD)**; (2) solve the mystery of part 01 and make sure our models will learn from the **brain**, not from something else.

In [ ]:
#@title (helper) MNE info
import mne
mne.set_log_level("ERROR")
info = mne_info(data)
c3, c4 = data["ch_names"].index("C3"), data["ch_names"].index("C4")

## 1. The brain signature: ERD
When you (imagine you) move a hand, the **mu rhythm (8–13 Hz)** over the motor cortex becomes *weaker*. This drop in power is called **ERD**. And the motor cortex is crossed, as we saw in part 01.

🤔 **Predict:** when people imagine the **LEFT** hand, where does mu power drop more?

In [ ]:
#@title 🤔 Your prediction
left_hand_erd = "Over the right hemisphere (C4)" #@param ["Over the left hemisphere (C3)", "Over the right hemisphere (C4)", "Both equally", "No change"]
share("left_hand_erd", left_hand_erd)

In [ ]:
#@title 🎛️ Choose the rhythm and draw the brain map (MNE topomap)
band = "mu (8-13 Hz)" #@param ["mu (8-13 Hz)", "beta (13-30 Hz)"]
lo, hi = (8, 13) if band.startswith("mu") else (13, 30)
Xb = bandpass(X, lo, hi, fs)
power = np.log((Xb[:, :, fs:] ** 2).mean(-1))        # log power per trial & electrode, from 1 s to 4 s
# Compare LEFT vs RIGHT *within each person*, then average over people
diff = np.mean([power[(subject == s) & (y == 0)].mean(0) - power[(subject == s) & (y == 1)].mean(0)
                for s in np.unique(subject)], axis=0)
fig, ax = plt.subplots(figsize=(4.6, 4.2))
im, _ = mne.viz.plot_topomap(diff, info, axes=ax, cmap="RdBu_r", show=False,
                             names=[n if n in ("C3", "C4") else "" for n in info.ch_names])
plt.colorbar(im, ax=ax, shrink=0.7, label="log power: LEFT − RIGHT")
ax.set_title(f"{band}: imagined LEFT minus imagined RIGHT")
plt.show()

👀 **What am I looking at?** A map of the head seen from above (nose up). For each electrode: power when imagining the **left** hand minus power when imagining the **right** hand, from 1 to 4 s.
🔵 blue = less power for *left* imagery · 🔴 red = less power for *right* imagery.

**What should you notice?** Blue over the **right** motor cortex (around C4) and red over the **left** one (around C3): imagining the left hand weakens the rhythm in the *right* hemisphere, and vice versa. The brain signature exists, and it is **spatial** (which electrode) and **spectral** (which rhythm).

In [ ]:
#@title ▶️ Check your prediction
print("You predicted:", left_hand_erd)
print(f"LEFT − RIGHT log-power at C3: {diff[c3]:+.3f}   at C4: {diff[c4]:+.3f}")
print("➡️ For LEFT-hand imagery the power is relatively lower at", "C4 (right hemisphere) ✅" if diff[c4] < diff[c3] else "C3")

In [ ]:
#@title ▶️ Mu power over time at C3 and C4
from scipy.signal import hilbert
Xmu = bandpass(X, 8, 13, fs)
env = np.abs(hilbert(Xmu[:, [c3, c4]], axis=-1)) ** 2
env = env / env.mean(axis=(0, 2), keepdims=True)                      # 1 = average power of that electrode
t = np.arange(X.shape[-1]) / fs
k = int(0.25 * fs)                                                      # 0.25 s smoothing
smooth = lambda v: np.convolve(v, np.ones(k) / k, mode="same")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, j, name in zip(axs, [0, 1], ["C3 (left hemisphere)", "C4 (right hemisphere)"]):
    for c in (0, 1):
        ax.plot(t, smooth(env[y == c, j].mean(0)), color=COLORS[CLASS_NAMES[c]], lw=2, label=f"imagined {CLASS_NAMES[c]}")
    ax.set(title=name, xlabel="time after target (s)", xlim=(0.2, 3.8))
axs[0].set_ylabel("relative mu power"); axs[0].legend()
plt.tight_layout(); plt.show()

👀 **What should you notice?** After the target appears, mu power drops for both hands (both lines go down: ERD). But it drops **more on the opposite side**: at **C4** the *left* line is lower, at **C3** the *right* line is lower. The effect is **small** — and this is the average of thousands of trials. In a *single* trial it is buried in noise. That is why we need machine learning.

## 2. 🕵️ The mystery of part 01
In part 01 the *average waveform* at C3 was different for left and right in the **first second**. ERD is a change in *power*, not a big synchronised wave, so something else is going on. Let's map that early difference on the head (MNE).

🤔 **Predict:** where on the head will the early left/right difference be strongest?

In [ ]:
#@title 🤔 Your prediction
early_difference = "Over the motor cortex (C3/C4)" #@param ["Over the motor cortex (C3/C4)", "At the back of the head (visual cortex)", "At the front, near the eyes", "Everywhere equally"]
share("early_difference", early_difference)

In [ ]:
#@title ▶️ Map of the average waveform difference LEFT − RIGHT (MNE)
Xslow = X - X.mean(-1, keepdims=True)                        # remove each trial's offset, keep slow waves
evoked_diff = mne.EvokedArray((Xslow[y == 0].mean(0) - Xslow[y == 1].mean(0)) * 1e-6, info, tmin=0)
fig = evoked_diff.plot_topomap(times=[0.3, 0.6, 1.5, 3.0], ch_type="eeg", show=False, time_unit="s",
                               cmap="RdBu_r", nrows=1, colorbar=True)
fig.suptitle("Average waveform: LEFT − RIGHT trials", fontweight="bold"); plt.show()
f7, f8 = data["ch_names"].index("F7"), data["ch_names"].index("F8")
print(f"At 0.2–0.5 s:  F7 (left of the eyes) {evoked_diff.data[f7, int(0.2 * fs):int(0.5 * fs)].mean()*1e6:+.0f} µV   "
      f"F8 (right of the eyes) {evoked_diff.data[f8, int(0.2 * fs):int(0.5 * fs)].mean()*1e6:+.0f} µV   C3 {evoked_diff.data[c3, int(0.2 * fs):int(0.5 * fs)].mean()*1e6:+.0f} µV")

👀 **What am I looking at?** Four snapshots of the head. Colour = average signal for left-target trials minus right-target trials.

**What should you notice?** At 0.3–0.6 s there is a strong **red/blue pair at the front**, on the left (F7) and right (F8) of the eyes, with **opposite signs**. That is the classic signature of the **eyes moving horizontally**: the eye is an electrical dipole, so when it turns left one side becomes positive and the other negative. People simply **looked at the target** on the left or on the right of the screen. It spreads to C3 as well. After ~1 s it fades.

**This is a confound**: it differs between left and right, but it is *not* the brain imagining a movement. A model could learn to classify **where the person looked** and we would believe it reads motor imagery.

## 3. Can a model cheat with the eyes?
Before class we trained EEGNet (a network you will meet in part 05) using **only the 14 electrodes at the front of the head**, which carry the eye signals but not the motor cortex. We did it with the whole trial (0–4 s) and without the first second (1–4 s), 3 times each.

🤔 **Predict:** with only the electrodes near the eyes and the whole trial (0–4 s), will it do better than chance (50%)?

In [ ]:
#@title 🤔 Your prediction
eyes_model = "No, ~50%" #@param ["No, ~50%", "A bit, ~55%", "Yes, clearly better"]
share("eyes_model", eyes_model)

In [ ]:
#@title ▶️ Results (precomputed: EEGNet, 8–30 Hz, 15 validation people, 3 seeds)
rows = load_precomputed("03_eyes")
combos = [("14 front electrodes (near the eyes)", "0-4 s"), ("14 front electrodes (near the eyes)", "1-4 s"),
          ("all 64 electrodes", "0-4 s"), ("all 64 electrodes", "1-4 s")]
labels = ["front (eyes), 0–4 s", "front (eyes), 1–4 s", "all 64, 0–4 s", "all 64, 1–4 s"]
vals = [[r["val_acc"] for r in rows if r["channels"] == c and r["window"] == w] for c, w in combos]
fig, ax = plt.subplots(figsize=(8, 3.2))
for k, v in enumerate(vals):
    ax.barh(k, np.mean(v), color=["#C44E52", "#B8C6DB", "#C44E52", "#4C72B0"][k], alpha=0.85)
    ax.scatter(v, [k] * len(v), color="k", s=14, zorder=3)
    ax.text(0.41, k, f"{np.mean(v):.2f}", va="center", color="white", fontweight="bold")
ax.axvline(0.5, color=GREY, ls="--"); ax.text(0.5, -0.7, " chance", color=GREY)
ax.set(yticks=range(4), yticklabels=labels, xlim=(0.4, 0.85), xlabel="validation accuracy"); ax.invert_yaxis()
plt.show()
print("You predicted:", eyes_model)

👀 **What should you notice?**
- With **only the electrodes near the eyes** and the whole trial, the model is clearly **above chance**: it is reading the eyes. 🚨
- Remove the **first second** and the same electrodes fall to **chance**: the eye movement is over, nothing is left to cheat with.
- With **all 64 electrodes**, the score drops when we remove the first second: part of the "good" result with 0–4 s was the eyes, not the brain.

✅ **Decision for the rest of the session: we use the window from 1 s to 4 s.** It costs us a lower score, but the score will be **honest**: it measures motor imagery.

💬 **Discuss:** in a real BCI (e.g., a wheelchair controlled by imagination) why would a model that reads eye movements be a problem? Have you seen other "Clever Hans" models (models that are right for the wrong reason)?

## 4. Does the brain signature appear in everybody?

In [ ]:
#@title ▶️ The ERD, person by person (1–4 s)
mu_pow = np.log((Xmu[:, :, fs:] ** 2).mean(-1))
lat = np.array([(mu_pow[(subject == s) & (y == 0)][:, c4].mean() - mu_pow[(subject == s) & (y == 1)][:, c4].mean())
                - (mu_pow[(subject == s) & (y == 0)][:, c3].mean() - mu_pow[(subject == s) & (y == 1)][:, c3].mean())
                for s in np.unique(subject)])
fig, ax = plt.subplots(figsize=(8, 3.4))
order = np.sort(lat)
ax.bar(range(len(order)), order, color=np.where(order < 0, "#4C72B0", "#C44E52"))
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="people (sorted)", ylabel="lateralisation index", title=f"Expected crossed pattern (bar below 0) in {np.mean(lat < 0):.0%} of the people")
plt.show()

👀 **What am I looking at?** One bar per person. Below zero (blue) = the person shows the expected crossed pattern. Above zero (red) = they do not.

**What should you notice?** Most people show it, but **not everybody**, and the strength varies a lot. This is well known in BCI research: for a fraction of users ("BCI inefficiency") motor-imagery BCIs barely work. Remember this when we look at the results per person in part 07.

## ✅ What you learned
- Imagining a hand **lowers mu power over the opposite hemisphere** (ERD). It is small and varies between people.
- The early left/right difference was the **eyes** looking at the target — a confound that a model can exploit.
- **Our pipeline now:** raw EEG → band-pass **8–30 Hz** → **window 1–4 s** → standardise each trial → model.

In [ ]:
#@title 🧹 Free memory before the next part
import gc
for _v in ['Xb', 'Xmu', 'Xslow', 'env', 'evoked_diff']:
    globals().pop(_v, None)
gc.collect(); print("✅ Memory freed")

# 04 · How to test honestly + a classic baseline ⚖️
**Goal:** decide how to split the data so that our accuracy tells the truth about **new users**, and build a classic BCI pipeline (CSP + LDA, with MNE) as the baseline to beat.

In [ ]:
#@title (helper) our pipeline so far (parts 02–03) + MNE
import mne
mne.set_log_level("ERROR")
info = mne_info(data)
Xf = bandpass(X, 8, 30, fs)[:, :, fs:]          # 8–30 Hz, window 1–4 s
print("Filtered 8–30 Hz, window 1–4 s:", Xf.shape)

## 1. Every brain has a fingerprint
🤔 **Predict:** we give a simple model ONE trial of 4 s. Can it tell **which of 20 people** it belongs to? (Guessing would give 5%.)

In [ ]:
#@title 🤔 Your prediction
who_is_it = "about 30%" #@param ["about 5% (chance)", "about 30%", "about 70%", "almost 100%"]
share("who_is_it", who_is_it)

In [ ]:
#@title ▶️ Test: guess the person from one trial
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
spec = spectral_features(Xf, fs, fmin=8, fmax=30)        # power spectrum of every electrode
people = np.unique(subject)[:20]
m = np.isin(subject, people)
tr_, te_ = trial_split(m.sum(), 0.3, seed=0)
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=3000)).fit(spec[m][tr_], subject[m][tr_])
acc = clf.score(spec[m][te_], subject[m][te_])
plot_scores(["guessing", "model"], [1 / 20, acc], chance=1 / 20, title="Which of 20 people is this trial from?")
plt.xlim(0, 1.05); plt.show()
print(f"You predicted: {who_is_it}.  Result: {acc:.0%}")

👀 **What should you notice?** EEG is extremely personal: from one 4-s trial the model recognises the person almost perfectly.

**Why does it matter?** If the same person appears in the training *and* the test data, a model can partly rely on "who is this" instead of "what is this person imagining", and the score no longer tells us how the BCI works for a **new user**.

✅ **Our rule: split by PERSON.** Every person is in exactly one set:
- **train** (70 people) → the model learns here
- **validation** (15 people) → we compare models and choose hyperparameters here
- **test** (21 people) → 🔒 **locked** until part 07. Used **once**, at the very end.

In [ ]:
#@title ▶️ Make the split and draw it
train, val, test = subject_split(subject)          # the function also checks that nobody is in two sets
people = np.unique(subject)
which = np.array([0 if train[subject == s][0] else 1 if val[subject == s][0] else 2 for s in people])
cols = np.array([[0.30, 0.45, 0.69], [0.33, 0.66, 0.41], [0.51, 0.45, 0.70]])
ncol = 27
grid = np.ones((int(np.ceil(len(people) / ncol)), ncol, 3))
for k, w in enumerate(which): grid[k // ncol, k % ncol] = cols[w]
fig, ax = plt.subplots(figsize=(10, 2.2))
ax.imshow(grid, aspect="equal")
ax.set_xticks(np.arange(-0.5, ncol, 1), minor=True); ax.set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", lw=2); ax.tick_params(which="both", length=0); ax.set_xticks([]); ax.set_yticks([])
for k, (name, n) in enumerate(zip(["train", "validation", "test (locked)"], [train, val, test])):
    ax.scatter([], [], color=cols[k], marker="s", s=80, label=f"{name}: {len(np.unique(subject[n]))} people, {n.sum()} trials")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5)); ax.set_title("One square = one person")
for s_ in ax.spines.values(): s_.set_visible(False)
plt.show()
print(f"Chance level in validation: {chance_level(y[val]):.1%}")

## 2. A classic BCI: CSP + LDA (MNE)
**CSP (Common Spatial Patterns)** looks for combinations of electrodes whose power is as **different** as possible between *left* and *right*. Then a simple linear classifier (**LDA**) decides.
It is the textbook motor-imagery pipeline and it is implemented in MNE. It only uses the **power** of the signal, which is exactly what ERD changes.

🎛️ **Decide:** the frequency band and the number of CSP components (window 1–4 s, as decided in part 03). Training uses only the **train** people; the score is on **validation** people.

In [ ]:
#@title 🎛️ Train CSP + LDA
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
csp_low = 8 #@param {type:"slider", min:4, max:12, step:1}
csp_high = 30 #@param {type:"slider", min:20, max:75, step:1}
n_components = 6 #@param {type:"slider", min:2, max:10, step:2}
Xcsp = bandpass(X, csp_low, csp_high, fs)[:, :, fs:]
if csp_high < 36:
    Xcsp = Xcsp[:, :, ::2]       # nothing left above 40 Hz → 80 Hz is enough (Nyquist), half the memory
Xcsp = Xcsp.astype(np.float64)
csp = CSP(n_components=n_components, reg="ledoit_wolf", log=True)
pipe = make_pipeline(csp, LDA()).fit(Xcsp[train], y[train])
csp_val = pipe.score(Xcsp[val], y[val])
plot_scores(["chance", "CSP + LDA"], [chance_level(y[val]), csp_val], title="Accuracy on VALIDATION people")
plt.show()
print(f"CSP + LDA ({csp_low}–{csp_high} Hz, 1–4 s, {n_components} components): {csp_val:.1%} on 15 new people")
share("csp_result", f"{csp_low}–{csp_high} Hz, {n_components} components → {csp_val:.1%}", leaderboard=True)

👀 **What should you notice?** Above chance, but far from perfect: around 60% on people the model has never seen. Decoding imagined movements of a *new* person, without calibrating on their data, is hard. This is our **baseline** — parts 05 and 06 try to beat it.

But is ~60% *really* better than chance, or could it be luck? → section 3.

In [ ]:
#@title ▶️ What did CSP learn? (MNE patterns)
fig = csp.plot_patterns(info, components=list(range(n_components)), ch_type="eeg", colorbar=False,
                        show=False, size=1.4, name_format="CSP %d")
plt.show()

👀 **What am I looking at?** Each map is a spatial pattern that CSP found useful (red/blue = electrodes that weigh a lot, with opposite signs). The first and the last components are the most discriminative.

**What should you notice?** Some patterns are **focal spots near C3 or C4**, over the motor cortex — the same places as the ERD map in part 03: the algorithm rediscovered the neuroscience by itself. Others are at the **back of the head** (visual alpha rhythm) or at the **edge of the head near the face** (probably muscles). CSP uses whatever differs between the classes — it does not know what is brain and what is not.

## 3. Sanity check: shuffled labels
🤔 **Predict:** we randomly shuffle the labels (left/right) and train the same pipeline again — 10 times, each with a different shuffle. What accuracy do you expect?

In [ ]:
#@title 🤔 Your prediction
shuffled_prediction = "about 50%" #@param ["about 50%", "about 60%", "about 70%"]
share("shuffled_prediction", shuffled_prediction)

In [ ]:
#@title ▶️ Train 10 times with shuffled labels (≈1 min)
rng = np.random.RandomState(0)
null = []
for i in range(10):
    y_sh = rng.permutation(y)
    p = make_pipeline(CSP(n_components=n_components, reg="ledoit_wolf", log=True), LDA()).fit(Xcsp[train], y_sh[train])
    null.append(p.score(Xcsp[val], y_sh[val]))
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hist(null, bins=np.linspace(0.40, 0.70, 31), color="#B8C6DB", label="shuffled labels (10 runs)")
ax.axvline(csp_val, color="#4C72B0", lw=3, label=f"real labels: {csp_val:.1%}")
ax.axvline(0.5, color=GREY, ls="--", label="chance")
ax.set(xlabel="validation accuracy", ylabel="runs", title="Is our score better than luck?"); ax.legend(loc="upper left")
plt.show()
print(f"You predicted {shuffled_prediction}. Shuffled labels: {np.mean(null):.1%} on average (range {min(null):.1%}–{max(null):.1%}).")
print(f"Real labels beat {np.mean(np.array(null) < csp_val):.0%} of the shuffled runs.")

👀 **What am I looking at?** The grey bars are what the *same* pipeline scores when the labels are random: a few points around 50%, just by luck. The blue line is the real score.

**What should you notice?**
- Shuffled labels give ≈50% ± a few points: that is the size of "luck" with 15 validation people.
- The real score is beyond all the shuffled ones → it is not luck (this is a *permutation test*).
- If shuffled labels had given ~60%, there would be a bug or a leak in the pipeline. **Always do this check.**

## ✅ What you learned
- EEG identifies people almost perfectly → **split by person** to measure what matters for new users.
- Train / validation / test 🔒. The test set is used only once, at the end.
- Classic baseline: **CSP + LDA ≈ 60%** on new people, clearly beyond the shuffled-label runs ✅.

In [ ]:
#@title 🧹 Free memory before the next part
import gc
for _v in ['Xf', 'Xcsp', 'spec']:
    globals().pop(_v, None)
gc.collect(); print("✅ Memory freed")

# 05 · Neural networks 🕸️
**Goal:** compare the networks you know — **MLP, CNN, RNN, LSTM, GRU** — plus two more specialised ones (**BiGRU, attention, EEGNet**) on exactly the same data, and learn to read training curves.

All scores here are on the **validation** people. The test set stays 🔒.

In [ ]:
#@title ▶️ Preprocess + split (exactly as in parts 02–04)
inp = prepare_inputs(data)          # band-pass 8–30 Hz, window 1–4 s, standardise each trial, split by person
train, val, test = inp["train"], inp["val"], inp["test"]
print(f"train: {train.sum()} trials ({len(np.unique(subject[train]))} people) | "
      f"validation: {val.sum()} ({len(np.unique(subject[val]))} people) | test 🔒: {test.sum()} ({len(np.unique(subject[test]))} people)")
print(f"sampling rate of the network inputs: {inp['fs']} Hz")
for k in ["features", "sequence", "image"]:
    print(f"  input '{k}': {inp[k].shape}")

💡 **Nyquist again.** After the 8–30 Hz filter nothing is left above 40 Hz, so we can keep **1 sample out of 2** (160 → 80 Hz) without losing any information: half the numbers, networks train twice as fast. That is why the 1–4 s window has 3 × 80 = **240** time steps.

👀 **Same data, three shapes.** Every network needs the input in a different shape:

| model | input | shape of one trial |
|---|---|---|
| **MLP** | mu and beta power of each electrode (the ERD of part 03, computed by us) | 64 electrodes × 2 bands → 128 numbers |
| **CNN1D, RNN, LSTM, GRU, BiGRU, attention** | the signal as a **sequence**: 240 time steps (1–4 s), each with 64 values | 240 × 64 |
| **EEGNet** | the signal as an **image**: electrodes × time | 64 × 240 × 1 |

All sequence models start with the same small convolution that shortens 240 steps to 60 (it makes training fast). After it comes the part that differs: convolution, recurrence or attention.

## 1. 🤔 Predict
Which model will be best on new people? And the worst?

In [ ]:
#@title 🤔 Your prediction
best_model_guess = "EEGNet" #@param ["MLP (band power)", "CNN1D", "SimpleRNN", "LSTM", "GRU", "BiGRU", "Attention", "EEGNet"]
worst_model_guess = "MLP (band power)" #@param ["MLP (band power)", "CNN1D", "SimpleRNN", "LSTM", "GRU", "BiGRU", "Attention", "EEGNet"]
share("best_model_guess", best_model_guess)
share("worst_model_guess", worst_model_guess)

## 2. 🎛️ Train a model
Choose a model and press ▶. If you work in a group, **each person trains a different model** and you compare.

⏱️ With a GPU it takes 10–60 s. Without a GPU, tick `use_precomputed` to load the result we trained before class with the same code.

In [ ]:
#@title 🎛️ Choose and train
model_name = "EEGNet" #@param ["MLP (band power)", "CNN1D", "SimpleRNN", "LSTM", "GRU", "BiGRU", "Attention", "EEGNet"]
rnn_summary = "average" #@param ["average", "last"]
epochs = 15 #@param {type:"slider", min:3, max:40, step:1}
use_precomputed = False #@param {type:"boolean"}
import time, keras
x = inp[MODEL_INPUT[model_name]]
if use_precomputed:
    rows = load_precomputed("05_models")
    summ = rnn_summary if model_name in ("SimpleRNN", "LSTM", "GRU", "BiGRU") else "average"
    r = [r for r in rows if r["model"] == model_name and r["summary"] == summ and r["seed"] == 0]
    if not r:
        raise ValueError("No precomputed result for this combination — untick use_precomputed")
    r = r[0]; history = r["history"]; y_pred = np.array(r["val_pred"]); n_params = r["params"]; secs = r["seconds"]
    print(f"Loaded precomputed result ({r['epochs']} epochs).")
else:
    keras.utils.set_random_seed(0)                       # same starting point for everybody
    model = build_model(model_name, inp, rnn_summary)
    t0 = time.time()
    history = model.fit(x[train], y[train], validation_data=(x[val], y[val]),
                        epochs=epochs, batch_size=64, verbose=0).history
    secs = time.time() - t0
    y_pred = model.predict(x[val], verbose=0).argmax(1); n_params = model.count_params()
val_acc = np.mean(y_pred == y[val])
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw=dict(width_ratios=[2, 1]))
plot_history(history, f"{model_name}: {n_params:,} parameters, {secs:.0f} s", chance=chance_level(y[val]), ax=axs[0])
plot_confusion(y[val], y_pred, f"Validation: {val_acc:.1%}", ax=axs[1])
plt.tight_layout(); plt.show()
if not use_precomputed:
    what = model_name + (f" ({rnn_summary})" if model_name in ("SimpleRNN", "LSTM", "GRU", "BiGRU") else "")
    share("nn_result", f"{what}, {epochs} epochs → {val_acc:.1%}", leaderboard=True)

👀 **How to read these figures**

*Left — learning curves.* One point per **epoch** (one pass over all training trials).
- **training** accuracy: how well the model fits the people it learns from.
- **validation** accuracy: how well it works on **15 new people**. This is the one that matters.
- If training keeps going up while validation stays flat or goes down → **overfitting** (memorising instead of learning).
- If both are still rising at the end → it could use **more epochs**.

*Right — confusion matrix* (validation): rows = what the person imagined, columns = what the model said. A good model has most trials on the diagonal. A model that answers "left" almost always would have a full left column — it is not really discriminating.

## 3. 🎛️ An architecture decision: how should an RNN summarise the sequence?
A recurrent network reads the 60 steps one by one and keeps a memory (the *hidden state*). At the end we must pass *something* to the classifier:
- **last**: only the memory after the last step.
- **average**: the average of the memories of all steps.

🤔 **Predict:** does it matter? Train `GRU` with `rnn_summary = last` and then with `average` in the cell above (or run the cell below for the precomputed comparison).

In [ ]:
#@title ▶️ GRU: "last" vs "average" (3 random seeds each, precomputed)
rows = load_precomputed("05_models")
res = {s: [r["val_acc"] for r in rows if r["model"] == "GRU" and r["summary"] == s] for s in ["last", "average"]}
fig, ax = plt.subplots(figsize=(6, 2.6))
for k, s in enumerate(["last", "average"]):
    ax.scatter(res[s], [k] * len(res[s]), s=70, color="#4C72B0", zorder=3)
    ax.text(np.mean(res[s]), k + 0.25, f"mean {np.mean(res[s]):.2f}", ha="center")
ax.axvline(0.5, color=GREY, ls="--"); ax.set(yticks=[0, 1], yticklabels=["GRU — last", "GRU — average"], xlim=(0.4, 0.8), ylim=(-0.6, 1.6), xlabel="validation accuracy")
plt.show()

👀 **What should you notice?** With **last**, the GRU stays at chance level: the information has to survive 60 steps of memory, and this is hard to learn. With **average**, every time step has a direct path to the classifier and the GRU learns. An architecture detail can make the difference between a model that works and one that does not.

## 4. The whole picture
Every model trained **3 times** with different random seeds (random initial weights and random order of the trials). The bar is the mean; the dots are the 3 runs.

In [ ]:
#@title ▶️ All models, 3 seeds each (precomputed with this same code)
rows = load_precomputed("05_models")
names = list(MODEL_INPUT)
means = [np.mean([r["val_acc"] for r in rows if r["model"] == n and r["summary"] == "average"]) for n in names]
order = np.argsort(means)[::-1]
fig, ax = plt.subplots(figsize=(8, 4.6))
for rank, k in enumerate(order):
    vals = [r["val_acc"] for r in rows if r["model"] == names[k] and r["summary"] == "average"]
    ax.barh(rank, np.mean(vals), color="#B8C6DB" if names[k] != model_name else "#4C72B0")
    ax.scatter(vals, [rank] * len(vals), color="k", s=14, zorder=3)
    ax.text(0.41, rank, f"{np.mean(vals):.2f}", va="center", fontsize=9)
csp = load_precomputed("csp")["val"]
ax.axvline(csp, color="#C44E52", ls="-", lw=1.5, label=f"CSP + LDA baseline ({csp:.2f})")
ax.axvline(chance_level(y[val]), color=GREY, ls="--", label="chance")
ax.legend(loc="lower right", frameon=True)
ax.set(yticks=range(len(names)), yticklabels=[names[k] for k in order], xlim=(0.4, 0.8), xlabel="validation accuracy (15 new people)",
       title="15 epochs, default settings"); ax.invert_yaxis()
plt.show()
print("Your predictions → best:", best_model_guess, "| worst:", worst_model_guess)

👀 **What should you notice?**
- All networks are **above chance**, but **none clearly beats the classic CSP + LDA line**. On people the model has never seen, with ~3 000 training trials, a good classic method is as strong as deep learning *out of the box*. Deep learning is not magic: it needs a lot of data, and here every person's brain is different.
- The **dots spread**: the same model trained again can move by several points just by chance. Differences of 1–2 points between models are **not** meaningful.
- **EEGNet** is tiny (≈2 000 parameters vs ≈25 000 for the RNNs) because it was designed for EEG: temporal filters (frequency) + spatial filters (electrodes), like the ERD of part 03. With default settings and 15 epochs it is not at its best — part 06 tries to fix that.

💬 **Discuss:** why can't the networks beat CSP here? (Think: amount of data, differences between people, size of the effect we saw in part 03.) What would change if we could record a few minutes of the *new* person first?

## ✅ What you learned
- Read learning curves: training vs validation, overfitting, under-training.
- Architecture details matter (GRU last vs average).
- Results vary between runs → compare models over **several seeds**.

# 06 · Hyperparameter search with KerasTuner 🎯
**Goal:** let the computer try many settings of a network and keep the best one — and check whether the winner is *really* better.

| | **parameters** | **hyperparameters** |
|---|---|---|
| what | the weights of the network | the settings *we* choose |
| examples | the numbers inside each filter | number of filters, layer size, dropout, learning rate, number of epochs |
| who sets them | learnt during training (`fit`) | chosen **before** training → searched here |

**Random search**: choose a few values for each hyperparameter, try random combinations, and keep the one with the best **validation** accuracy. The test set stays 🔒.

In [ ]:
#@title (helper) table and plot functions for the search results
import pandas as pd
def plot_effects(table, title):
    hps = [c for c in table.columns if c != "val_accuracy"]
    fig, axs = plt.subplots(1, len(hps), figsize=(3.1 * len(hps), 3.2), sharey=True)
    rng = np.random.RandomState(0)
    for ax, hp in zip(np.atleast_1d(axs), hps):
        vals = [str(v) for v in table[hp]]
        cats = sorted(set(vals))
        xs = [cats.index(v) + rng.uniform(-0.08, 0.08) for v in vals]
        ax.scatter(xs, table["val_accuracy"], s=60, color="#4C72B0", alpha=0.8)
        ax.set(xticks=range(len(cats)), xticklabels=cats, title=hp, xlim=(-0.5, len(cats) - 0.5))
        ax.axhline(0.5, color=GREY, ls="--", lw=1)
    np.atleast_1d(axs)[0].set_ylabel("validation accuracy")
    fig.suptitle(title, fontweight="bold"); plt.tight_layout(); plt.show()
def show_table(trials):
    return pd.DataFrame(trials).style.background_gradient(subset=["val_accuracy"], cmap="Greens").format({"val_accuracy": "{:.3f}"})
print("✅ helpers ready")

## 1. 🤔 First: is one run enough?
If you train the **same** model with the **same** settings twice, will you get the same validation accuracy?

In [ ]:
#@title 🤔 Your prediction
same_twice = "Exactly the same" #@param ["Exactly the same", "±1 point", "±5 points or more"]
share("same_twice", same_twice)

In [ ]:
#@title ▶️ EEGNet with default settings, trained 3 times (precomputed)
seeds = load_precomputed("06_seeds")
d = [r["val_acc"] for r in seeds if r["config"] == "default"]
fig, ax = plt.subplots(figsize=(6, 1.8))
ax.scatter(d, [0] * len(d), s=80, color="#4C72B0", zorder=3)
ax.axvline(0.5, color=GREY, ls="--"); ax.set(yticks=[], ylim=(-1, 1), xlim=(0.45, 0.85), xlabel="validation accuracy",
                                              title="EEGNet trained 3 times: same model, same data, different random seed")
plt.show()
print(f"You predicted: {same_twice}. Range: {min(d):.2f} – {max(d):.2f}")

👀 **What should you notice?** Same code, same data — only the random initial weights and the order of the trials change — and the result moves by several points. So a difference of 2–3 points between two settings in **one** run may be pure luck. We come back to this at the end.

## 2. Exercise A — tune a recurrent network (GRU / LSTM)
🎛️ **Decide** the search space: write the options separated by commas. The tuner tries `max_trials` random combinations.

💡 Tip: more options × few trials = most combinations are never tried. Start small.

In [ ]:
#@title 🎛️ Exercise A — search space for the recurrent network
rnn_types = "GRU, LSTM" #@param {type:"string"}
units_options = "16, 32, 64" #@param {type:"string"}
bidirectional_options = "no, yes" #@param {type:"string"}
summary_options = "last, average" #@param {type:"string"}
lr_options = "0.01, 0.001" #@param {type:"string"}
max_trials = 6 #@param {type:"slider", min:2, max:12, step:1}
epochs = 10 #@param {type:"slider", min:3, max:20, step:1}
use_precomputed = False #@param {type:"boolean"}

This is the only code we ask you to **read** today. It is the heart of KerasTuner: a function that builds a model, where each setting is replaced by `hp.Choice(name, options)`.

In [ ]:
import keras
import keras_tuner as kt
from keras import layers

def as_list(text, cast):
    return [cast(v.strip()) for v in text.split(",")]

def build_rnn_model(hp):
    """KerasTuner calls this function once per trial. Every hp.Choice is a
    hyperparameter: the tuner picks one of the options each time."""
    kind    = hp.Choice("rnn_type",      as_list(rnn_types, str))
    units   = hp.Choice("units",         as_list(units_options, int))
    bidir   = hp.Choice("bidirectional", as_list(bidirectional_options, str))   # "no" or "yes"
    summary = hp.Choice("summary",       as_list(summary_options, str))
    lr      = hp.Choice("learning_rate", as_list(lr_options, float))
    n_times, n_chans = inp["sequence"].shape[1:]
    return build_rnn(kind, bidirectional=(bidir == "yes"), units=units, summary=summary, lr=lr,
                     n_times=n_times, n_chans=n_chans)

def run_search(build_fn, x_train, x_val, max_trials, epochs, name):
    tuner = kt.RandomSearch(build_fn, objective="val_accuracy", max_trials=max_trials,
                            seed=1, directory="tuning", project_name=name, overwrite=True)
    tuner.search(x_train, y[train], validation_data=(x_val, y[val]),
                 epochs=epochs, batch_size=64, verbose=2)   # one line per epoch
    trials = [dict(t.hyperparameters.values, val_accuracy=t.score)
              for t in tuner.oracle.trials.values() if t.score is not None]
    return sorted(trials, key=lambda r: -r["val_accuracy"])

print("✅ build_rnn_model and run_search are defined")

In [ ]:
#@title ▶️ Run the search (Exercise A)
if use_precomputed:
    trials_rnn = load_precomputed("06_tuner_rnn")
    print("Loaded precomputed search (default search space).")
else:
    trials_rnn = run_search(build_rnn_model, inp["sequence"][train], inp["sequence"][val], max_trials, epochs, "rnn")
table_rnn = pd.DataFrame(trials_rnn)
if not use_precomputed and trials_rnn:
    best = trials_rnn[0]
    share("tuner_rnn", ", ".join(f"{k}={v}" for k, v in best.items() if k != "val_accuracy") + f" → {best['val_accuracy']:.1%}", leaderboard=True)
show_table(trials_rnn)

In [ ]:
#@title ▶️ Which hyperparameters mattered? (Exercise A)
plot_effects(table_rnn, "Exercise A — each dot is one trial")

👀 **What am I looking at?** One panel per hyperparameter. Each dot is one trial of the search, placed under the value it used. If the dots under one value are clearly higher, that value tends to help.

**What should you notice?** With only a handful of trials, each value was tried a few times and mixed with the others, so read it as a *hint*, not as proof. Did `summary = last` vs `average` show up again, as in part 05?

## 3. Exercise B — tune EEGNet
EEGNet's main hyperparameters:
- **F1** — number of temporal filters (how many different "rhythms" it looks for)
- **kernel** — length of each temporal filter in samples (the networks see 80 samples per second — part 05 — so 80 samples = 1 s; longer kernels can resolve lower frequencies)
- **dropout** — fraction of units switched off at random during training (fights overfitting)
- **learning rate** — size of each update step

In [ ]:
#@title 🎛️ Exercise B — search space for EEGNet
F1_options = "4, 8, 16" #@param {type:"string"}
kernel_options = "10, 20, 40" #@param {type:"string"}
dropout_options = "0.25, 0.5" #@param {type:"string"}
lr_options_eegnet = "0.01, 0.001" #@param {type:"string"}
max_trials_eegnet = 6 #@param {type:"slider", min:2, max:12, step:1}
epochs_eegnet = 12 #@param {type:"slider", min:3, max:25, step:1}
use_precomputed_eegnet = False #@param {type:"boolean"}

In [ ]:
def build_eegnet_model(hp):
    F1      = hp.Choice("F1",            as_list(F1_options, int))          # number of temporal filters
    kernel  = hp.Choice("kernel",        as_list(kernel_options, int))      # filter length in samples (80 = 1 s)
    dropout = hp.Choice("dropout",       as_list(dropout_options, float))   # regularisation
    lr      = hp.Choice("learning_rate", as_list(lr_options_eegnet, float)) # step size of the optimiser
    n_chans, n_times = inp["image"].shape[1:3]
    return build_eegnet(n_chans, n_times, F1=F1, kernel=kernel, dropout=dropout, lr=lr)

print("✅ build_eegnet_model is defined")

In [ ]:
#@title ▶️ Run the search (Exercise B)
if use_precomputed_eegnet:
    trials_eeg = load_precomputed("06_tuner_eegnet")
    print("Loaded precomputed search (default search space).")
else:
    trials_eeg = run_search(build_eegnet_model, inp["image"][train], inp["image"][val], max_trials_eegnet, epochs_eegnet, "eegnet")
table_eeg = pd.DataFrame(trials_eeg)
plot_effects(table_eeg, "Exercise B — EEGNet")
if not use_precomputed_eegnet and trials_eeg:
    best = trials_eeg[0]
    share("tuner_eegnet", ", ".join(f"{k}={v}" for k, v in best.items() if k != "val_accuracy") + f" → {best['val_accuracy']:.1%}", leaderboard=True)
show_table(trials_eeg)

## 4. 🤔 Is the winner really better?
The tuner reports the **best** validation accuracy of each trial, and we then pick the **best** trial. Picking the maximum of noisy numbers is optimistic: part of the winner's score is luck.

The honest check: train the default and the winning settings **several times** and compare the spreads.

In [ ]:
#@title ▶️ Default vs tuned EEGNet, 3 seeds each (precomputed)
seeds = load_precomputed("06_seeds")
fig, ax = plt.subplots(figsize=(7, 2.6))
cfg_txt = lambda c: ", ".join(f"{k}={v}" for k, v in c.items()) or "F1=8, kernel=40, dropout=0.5, lr=0.001"
labels = {c: f"{c} ({cfg_txt([r for r in seeds if r['config'] == c][0]['settings'])})" for c in ["default", "tuned"]}
for k, cfg in enumerate(["default", "tuned"]):
    v = [r["val_acc"] for r in seeds if r["config"] == cfg]
    ax.scatter(v, [k] * len(v), s=80, color=["#8A8A8A", "#4C72B0"][k], zorder=3)
    ax.text(np.mean(v), k + 0.28, f"mean {np.mean(v):.2f}", ha="center")
ax.axvline(0.5, color=GREY, ls="--")
ax.set(yticks=[0, 1], yticklabels=[labels["default"], labels["tuned"]], xlim=(0.45, 0.85), ylim=(-0.6, 1.7), xlabel="validation accuracy")
plt.show()

In [ ]:
#@title ▶️ The verdict, computed
d = np.array([r["val_acc"] for r in seeds if r["config"] == "default"])
tu = np.array([r["val_acc"] for r in seeds if r["config"] == "tuned"])
print(f"default: mean {d.mean():.3f}  (runs: {', '.join(f'{v:.2f}' for v in d)})")
print(f"tuned  : mean {tu.mean():.3f}  (runs: {', '.join(f'{v:.2f}' for v in tu)})")
gain, noise = 100 * (tu.mean() - d.mean()), 100 * np.concatenate([d - d.mean(), tu - tu.mean()]).std()
print(f"Tuned is better in {np.sum(tu > d)} of {len(d)} paired runs; mean gain {gain:+.1f} points; seed-to-seed noise ≈ ±{noise:.1f} points.")
if tu.min() > d.max():
    print("✅ Every tuned run is above every default run: a real improvement.")
elif abs(gain) < noise:
    print("❌ No real gain: the difference is smaller than the noise between runs. The search 'winner' was partly luck.")
else:
    print("⚠️ The clouds overlap: maybe a small gain, but we would need more runs to be sure.")

👀 **What should you notice?** Compare the **distance between the two clouds** with the **spread inside each cloud**. A gain is convincing only if it is clearly bigger than the noise between runs. With 3 seeds we can see a trend; for a paper we would use more seeds and report the mean ± standard deviation.

💬 **Discuss:** in your own search, which settings won? Would you trust a difference of 1 point? What would you do with more time? (Hint: more trials, more seeds, a smarter search such as `kt.BayesianOptimization` or **Optuna**.)

✍️ **Write down your best EEGNet settings** — you will need them in part 07.

## ✅ What you learned
- Parameters are learnt; **hyperparameters** are chosen — and can be searched automatically with **KerasTuner**.
- The search uses **validation** people only; the test set stays 🔒.
- The best trial is optimistic → confirm with **several seeds** before you believe it.

# 07 · The final test 🔓
**Goal:** train the final model once, open the 🔒 **test set** once, and see how our BCI works for **21 people nobody has used before**.

Rules of the game: we already made **all** decisions (filter, split, model, hyperparameters) with train and validation. Now we only measure. If we changed something after seeing the test result, the test would stop being honest.

## 1. 🎛️ Your final settings
Write the EEGNet settings you chose in part 06 (the defaults below are the winners of the precomputed search). ⚠️ Decide **before** running: no changes after seeing the test result.

In [ ]:
#@title 🎛️ Final EEGNet settings
F1 = 8 #@param {type:"integer"}
kernel = 40 #@param {type:"integer"}
dropout = 0.25 #@param {type:"number"}
learning_rate = 0.01 #@param {type:"number"}
epochs = 15 #@param {type:"integer"}
use_precomputed = False #@param {type:"boolean"}

🤔 **Predict:** compare with your best **validation** accuracy in part 06. What will the **test** accuracy be?

In [ ]:
#@title 🤔 Your prediction
test_prediction = "About the same" #@param ["Higher", "About the same", "A bit lower", "Much lower"]
share("test_prediction", test_prediction)

In [ ]:
#@title ▶️ Train once, test once
import keras
if use_precomputed:
    r = load_precomputed("07_final")
    val_acc, test_acc = r["val_acc"], r["test_acc"]
    test_pred = np.array(r["test_pred"]); history = r["history"]
    print("Loaded precomputed final model:", r["config"])
else:
    keras.utils.set_random_seed(0)
    final = build_eegnet(*inp["image"].shape[1:3], F1=F1, kernel=kernel, dropout=dropout, lr=learning_rate)
    history = final.fit(inp["image"][train], y[train], validation_data=(inp["image"][val], y[val]),
                        epochs=epochs, batch_size=64, verbose=0).history
    val_acc = np.mean(final.predict(inp["image"][val], verbose=0).argmax(1) == y[val])
    test_pred = final.predict(inp["image"][test], verbose=0).argmax(1)       # 🔓 the only time we touch the test set
    test_acc = np.mean(test_pred == y[test])
csp = load_precomputed("csp")
fig, axs = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw=dict(width_ratios=[2, 1]))
plot_scores(["chance", "CSP + LDA (test)", "EEGNet (validation)", "EEGNet (TEST)"],
            [chance_level(y[test]), csp["test"], val_acc, test_acc], chance=chance_level(y[test]),
            title="Final results", ax=axs[0], highlight="EEGNet (TEST)")
plot_confusion(y[test], test_pred, f"Test: {test_acc:.1%}", ax=axs[1])
plt.tight_layout(); plt.show()
print(f"You predicted: {test_prediction}.  Validation {val_acc:.1%} → Test {test_acc:.1%}")
if not use_precomputed:
    share("final_test", f"EEGNet F1={F1}, kernel={kernel}, dropout={dropout}, lr={learning_rate} → validation {val_acc:.1%}, TEST {test_acc:.1%}", leaderboard=True)

👀 **What should you notice?**
- Validation vs test: we chose the settings *because* they were good on the validation people, so the validation score tends to be **optimistic**. The test score is the honest number we would report in a paper.
- Compare EEGNet and CSP + LDA on the **same 21 people**: is the difference bigger than the seed-to-seed noise we saw in part 06?
- Everything is far from 100%: decoding imagined movements of a person the model has **never seen** — no calibration, eye movements removed — is genuinely hard.

## 2. Does it work for everybody?
Each test person has ~45 trials. With 45 trials, ≥ 29 correct (64%) is unlikely to happen by guessing (p < 0.05).

In [ ]:
#@title ▶️ Accuracy per test person
from scipy.stats import binom
people = np.unique(subject[test])
acc_p = np.array([np.mean(test_pred[subject[test] == p] == y[test][subject[test] == p]) for p in people])
n_p = int(np.median([np.sum(subject[test] == p) for p in people]))
threshold = (binom.ppf(0.95, n_p, 0.5) + 1) / n_p     # smallest score with p < 0.05 under pure guessing
order = np.argsort(acc_p)[::-1]
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(range(len(people)), acc_p[order], color=np.where(acc_p[order] >= threshold, "#55A868", "#B8C6DB"))
ax.axhline(threshold, color="#C44E52", ls="--"); ax.text(len(people) - 0.5, threshold + 0.01, f"better than guessing (p<0.05): {threshold:.0%}", ha="right", color="#C44E52")
ax.axhline(0.5, color=GREY, ls=":")
ax.set(xticks=range(len(people)), xticklabels=[f"P{p}" for p in people[order]], ylim=(0.2, 1), ylabel="accuracy", title=f"The BCI works (green) for {np.sum(acc_p >= threshold)} of {len(people)} new people")
plt.xticks(rotation=90, fontsize=8); plt.show()

👀 **What should you notice?** The average hides a huge spread: for some people the model is very good, for others it is at chance. This matches what we saw in part 03 (not everybody shows a clear ERD). Real BCIs therefore usually **calibrate** with a few minutes of data from each new user.

💬 **Discuss:** how would you improve the BCI for the people in grey? (More data from them? Calibration? Training the user?)

## 3. ⭐ Bonus: attention
Attention lets the network decide **which moments of the trial** to focus on. We evaluate it on **validation** only — the test set has already been used for our final decision, so we do not use it to explore new models.

In [ ]:
#@title ▶️ Bonus: train an attention model (validation only)
use_precomputed_attention = False #@param {type:"boolean"}
if use_precomputed_attention:
    att_val = load_precomputed("07_final")["attention_val"]
else:
    keras.utils.set_random_seed(0)
    att = build_attention(*inp["sequence"].shape[1:])
    att.fit(inp["sequence"][train], y[train], epochs=15, batch_size=64, verbose=0)
    att_val = np.mean(att.predict(inp["sequence"][val], verbose=0).argmax(1) == y[val])
print(f"Attention model, validation: {att_val:.1%}  (tuned EEGNet, validation: {val_acc:.1%})")

## ✅ The whole pipeline
**raw EEG** → **band-pass 8–30 Hz** → **window 1–4 s** (no eye movements) → **standardise each trial** → **split by person** (train / validation / test 🔒) → **baseline** CSP + LDA → **networks** → **hyperparameter search** on validation → **one final test**

### Take-home messages
1. Look at the data and the neuroscience first (ERD over C3/C4) — and look for what else could explain the result (**the eyes**: 0.74 → 0.58).
2. Split by **person** when the question is "will it work for a new user?".
3. Always compare with **chance** and a **simple baseline**, and run a **shuffled-label** check.
4. Search hyperparameters on **validation**, confirm with **several seeds** (the search winner can be luck), open the **test** set once.
5. An honest ~60% on new people is worth more than an impressive number you cannot trust.

**Dataset:** Schalk, G., McFarland, D.J., Hinterberger, T., Birbaumer, N., Wolpaw, J.R. (2004). *BCI2000: A General-Purpose Brain-Computer Interface (BCI) System.* IEEE TBME 51(6):1034–1043 · Goldberger et al. (2000) *PhysioNet.* · **EEGNet:** Lawhern et al. (2018), *J. Neural Eng.* 15:056013.